#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[3]
CATENETS_DIR = ROOT / "experiments" / "supplementary" / "catenets_custom"
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# install dependencies - only once
%pip install wandb loguru jax ott-jax gdown
%pip install -e "{CATENETS_DIR}"

In [ ]:
# custom library imports - restart kernel after installation if needed
from catenets.models.torch.representation_nets import PairNet

In [ ]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### data

In [ ]:
# set dataset parameters
dataset = 'synthetic'
train_size = 1000

In [ ]:
# set confounders
confounders = ['x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9']
input_dim = len(confounders)

In [ ]:
# read
data_path = ROOT / "data" / "datasets" / f"{dataset}.csv"
df = pd.read_csv(data_path, index_col=0)

#### helpers

In [ ]:
def init_pairnet(input_dim, params, seed):
    return PairNet(
        # standard init
        n_unit_in=input_dim,             
        binary_y=False,
        seed=seed,
        n_iter_print=1,
        progress_bar=True,

        # model capacity
        n_layers_r=2,                    
        n_layers_out=2,                 
        n_units_r=int(params['hidden_dim']),
        n_units_out=int(params['hidden_dim']),

        # tuned parameters
        lr=params['learning_rate'],                
        weight_decay=params['weight_decay'],
        batch_size=int(params['batch_size']),

        # early stopping
        n_iter=50,                      
        early_stopping=True,
        patience=5,
        val_split_prop=0.2,

        # model-specific settings
        num_cfz=3,                        
        dynamic_phi=False,
        avg_objective=True,
        batch_norm=True,
        pair_sm_temp=1.0,
        pair_dist='euc',
        pair_pcs_dist=False,
        pair_drop_frac=0.0,
        pair_det=False,
        pair_arbitrary_pairs=False,
        nonlin='relu',
        dropout=False)

#### train and store

In [ ]:
# load configs
configs = pd.read_csv(f'./configs/pairnet.csv', index_col=0)

# set configs
row = configs.iloc[0]
params = dict(
    hidden_dim=int(row["hidden_dim"]),
    learning_rate=float(row["lr"]),
    weight_decay=float(row["weight_decay"]),
    batch_size=int(row["batch_size"]),
    max_epochs=50,
    patience=5)

In [ ]:
# set directories
out_dir = f'./chkpts/'
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# loop over seeds
for seed in range(5):

    # track progress
    print(f" -> Seed {seed}, size {train_size}")
    set_seed(seed)

    # set output dir
    ckpt_dir = Path(out_dir) / f"size_{train_size}" / f"seed_{seed}"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # get training data - .fit() uses internal train val split so merge
    _, _, train_df, val_df, _ = make_splits(df=df, train_size=train_size, seed=seed)
    train_df = pd.concat([train_df, val_df])

    # extract arrays
    X = train_df[confounders].to_numpy(dtype=np.float32)
    t = train_df["T"].to_numpy(dtype=np.float32)   
    y = train_df["Y"].to_numpy(dtype=np.float32) 

    # init model
    model = init_pairnet(input_dim, params, seed)

    # train
    model, loss = model.fit(X, y, t)
    
    # checkpoint
    torch.save(model.state_dict(), ckpt_dir / "PairNet.pt")

#### evaluation

In [ ]:
def load_pairnet(train_size, params, seed, confounders, device):
    input_dim = len(confounders)

    # set checkpoint path
    ckpt_path = ROOT / "experiments" / "supplementary" / "baselines" / "catenets_baselines" / "chkpts" / f"size_{train_size}" / f"seed_{seed}" / "PairNet.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")

    # load model checkpoint
    model = init_pairnet(input_dim=input_dim, params=params, seed=seed,)
    model.load_state_dict(torch.load(ckpt_path, weights_only=True))
    model.eval()
    
    return model

In [ ]:
def get_estimates_pairnet(model, confounders, test_df):
    X = test_df[confounders].to_numpy(dtype=np.float32)
    cate_te, mu0_te, mu1_te = pairnet.predict(X, return_po=True)
    test_df["pairnet_hat"] = cate_te.detach().cpu().numpy()
    return test_df

In [ ]:
def compute_metrics_pairnet(eval_df):
    required = {"pairnet_hat", "cate"}

    # check if estimates are available
    missing = required - set(eval_df.columns)
    if missing:
        raise ValueError(f"eval_df is missing required columns: {sorted(missing)}")

    specs = [("PairNet", "pairnet_hat")]
    rows = []
    for name, score_col in specs:
        ranked = eval_df.sort_values(score_col, ascending=False).copy()

        rows.append({
            "model": name,
            "autoc": autoc(ranked),
            "policy_value": policy_value(ranked)})

    return pd.DataFrame(rows)

In [ ]:
# init collector
all_metrics = []

# loop over seeds and sizes
for seed in range(5):
    for size in [100, 250, 500, 1000, 2000]:

        # get testing data
        _, _, _, _, test_df = make_splits(df=df, train_size=size, seed=seed)

        # load model
        pairnet = load_pairnet(size, params, seed, confounders, device)
        df_eval = get_estimates_pairnet(pairnet, confounders, test_df)
        df_metrics = compute_metrics_pairnet(df_eval)

        # store
        df_metrics["size"] = size
        df_metrics["seed"] = seed
        all_metrics.append(df_metrics)

# summarize
df_all = pd.concat(all_metrics, ignore_index=True)
summary = (df_all.groupby(["size", 'model']).agg(['mean', 'std']).reset_index())